In [ ]:
import torch
import torch.nn as nn
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
seed = 0
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [ ]:
class EmpiricalVelocityField(nn.Module):
    def __init__(self, samples: torch.Tensor, learnable: bool = False) -> None:
        super().__init__()
        samples = samples.detach().clone()

        if learnable:
            self.samples = nn.Parameter(samples)
        else:
            self.register_buffer("samples", samples)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        samples = self.samples

        sq_dist = torch.cdist(x, t * samples).square()
        weights = torch.softmax(-sq_dist / (2 * (1 - t)**2), dim=-1)

        return (weights @ samples - x) / (1 - t)

In [ ]:
def reconstruct(
        true_field: EmpiricalVelocityField,
        recon_field: EmpiricalVelocityField,
        steps: int = 5000,
        batch_size: int = 128,
        lr: float = 0.05,
    ):
    optimizer = torch.optim.Adam(recon_field.parameters(), lr)

    device = recon_field.samples.device
    d = recon_field.samples.shape[1]

    for step in range(steps):
        t = torch.rand((), device=device) * 0.3 + 0.35
        x = torch.empty(batch_size, d, device=device).uniform_(-2.5, 2.5)

        with torch.no_grad():
            target_v = true_field(x, t)

        pred_v = recon_field(x, t)
        loss = (pred_v - target_v).square().mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 1000 == 0:
            print(f"Step {step}: Velocity MSE = {loss.item():.6f}")


In [ ]:
def sample_gmm(n, weights, means, stds):
    k = torch.multinomial(weights, n, replacement=True)
    return means[k] + stds[k, None] * torch.randn(n, means.shape[1])

In [ ]:
weights = torch.tensor([0.5, 0.5])
means = torch.tensor([[1.0], [-1.0]])
stds = torch.tensor([0.3, 0.3])

# Reconstruct against Closed-form Empirical Velocity Field 

## 1D 1/2 Unknown Data

In [ ]:
# Data
n = 2
data = sample_gmm(n, weights, means, stds)

# Fields
true_field = EmpiricalVelocityField(data)

candidates = torch.cat([
    data[:1],
    torch.randn_like(data[1:])
])

recon_field = EmpiricalVelocityField(candidates, learnable=True)

mask = torch.ones_like(recon_field.samples)
mask[0] = 0

recon_field.samples.register_hook(lambda grad: grad * mask)
reconstruct(
    true_field,
    recon_field,
    steps=5000,
    batch_size=128,
    lr=0.05
)

In [ ]:
print("Known sample:       ", data[0].item())
print("True unknown:       ", data[1].item())
print("Reconstructed:      ", recon_field.samples[1].item())

error = (recon_field.samples[1] - data[1]).abs().item()
print("Reconstruction error:", error)

## 1D 2/2 Unknown Data 

In [ ]:
# Data
n = 2
data = sample_gmm(n, weights, means, stds)

# Fields
true_field = EmpiricalVelocityField(data)

candidates = torch.randn_like(data)
recon_field = EmpiricalVelocityField(candidates, learnable=True)

reconstruct(
    true_field,
    recon_field,
    steps=5000,
    batch_size=128,
    lr=0.05
)

In [ ]:
true_samples = data.detach().sort(dim=0).values
recon_samples = recon_field.samples.detach().sort(dim=0).values

print("Ground truth:  ", true_samples.flatten().tolist())
print("Reconstructed: ", recon_samples.flatten().tolist())

error = (recon_samples - true_samples).abs().mean()
print("Reconstruction MAE:", error.item())

## 1D 5/5 Unknown Data

In [ ]:
# Data
n = 5
data = sample_gmm(n, weights, means, stds)

# Fields
true_field = EmpiricalVelocityField(data)

candidates = torch.randn_like(data)
recon_field = EmpiricalVelocityField(candidates, learnable=True)

reconstruct(
    true_field,
    recon_field,
    steps=10000,
    batch_size=128,
    lr=0.05
)

In [ ]:
true_samples = data.detach().sort(dim=0).values
recon_samples = recon_field.samples.detach().sort(dim=0).values

error = (recon_samples - true_samples).abs().mean()

print("Ground truth:  ", true_samples.flatten().tolist())
print("Reconstructed: ", recon_samples.flatten().tolist())
print("Reconstruction MAE:", error.item())

## 1D 10/10 Unknown Data

In [ ]:
# Data
n = 10
data = sample_gmm(n, weights, means, stds)

# Fields
true_field = EmpiricalVelocityField(data)

candidates = torch.randn_like(data)
recon_field = EmpiricalVelocityField(candidates, learnable=True)

reconstruct(
    true_field,
    recon_field,
    steps=20000,
    batch_size=512,
    lr=0.01
)

In [ ]:
true_samples = data.detach().sort(dim=0).values
recon_samples = recon_field.samples.detach().sort(dim=0).values

error = (recon_samples - true_samples).abs().mean()

print("Ground truth:  ", true_samples.flatten().tolist())
print("Reconstructed: ", recon_samples.flatten().tolist())
print("Reconstruction MAE:", error.item())

## 2D

In [ ]:
torch.manual_seed(0)
means = torch.tensor([
    [-1.0, -1.0],
    [ 1.0,  1.0]
])

data = torch.cat([
    means[0] + 0.3 * torch.randn(5, 2),
    means[1] + 0.3 * torch.randn(5, 2)
], dim=0)

true_field = EmpiricalVelocityField(data)

torch.manual_seed(1)
recon_field = EmpiricalVelocityField(
    torch.randn_like(data),
    learnable=True
)

reconstruct(
    true_field,
    recon_field,
    steps=15000,
    batch_size=512,
    lr=0.01
)

In [ ]:
true_samples = true_field.samples.detach().cpu()
recon_samples = recon_field.samples.detach().cpu()

dist = torch.cdist(true_samples, recon_samples)
row, col = linear_sum_assignment(dist.numpy())

errors = dist[row, col]

print("Reconstruction Mean Distance:", errors.mean().item())
print("Reconstruction Max Distance: ", errors.max().item())

In [ ]:
true = true_field.samples.detach().cpu()
recon = recon_field.samples.detach().cpu()

plt.scatter(true[:, 0], true[:, 1], label="Ground Truth")
plt.scatter(recon[:, 0], recon[:, 1], marker="x", label="Reconstructed")

plt.axis("equal")
plt.legend()
plt.show()

In [ ]:
torch.manual_seed(0)

n = 100
d = 8
std = 0.3

means = torch.zeros(4, d)
means[:, :2] = torch.tensor([
    [-1.0, -1.0],
    [-1.0,  1.0],
    [ 1.0, -1.0],
    [ 1.0,  1.0]
])

labels = torch.arange(n) % 4
data = means[labels] + std * torch.randn(n, d)

In [ ]:
true_field = EmpiricalVelocityField(data)

torch.manual_seed(1)
recon_field = EmpiricalVelocityField(
    torch.randn_like(data),
    learnable=True
)

reconstruct(
    true_field,
    recon_field,
    steps=15000,
    batch_size=512,
    lr=0.01
)

In [ ]:
true_samples = data.detach()
recon_samples = recon_field.samples.detach()

dist = torch.cdist(true_samples, recon_samples)
row, col = linear_sum_assignment(dist.cpu().numpy())

errors = dist[row, col]

print("Reconstruction Mean Distance:", errors.mean().item())
print("Reconstruction Max Distance: ", errors.max().item())

In [ ]:
combined = torch.cat([true_samples, recon_samples], dim=0)

pca = PCA(n_components=2)
projected = pca.fit_transform(combined.numpy())

true_2d = projected[:len(true_samples)]
recon_2d = projected[len(true_samples):]

plt.figure(figsize=(6, 6))
plt.scatter(true_2d[:, 0], true_2d[:, 1], label="Ground Truth")
plt.scatter(recon_2d[:, 0], recon_2d[:, 1], marker="x", label="Reconstructed")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.axis("equal")
plt.legend()
plt.show()

# Reconstruct against Learned Velocity Field

In [ ]:
class VelocityField(nn.Module):
    def __init__(self, d: int, hidden_dim: int = 128):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, d)
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t = torch.as_tensor(t, device=x.device, dtype=x.dtype)
        t = t.reshape(-1, 1).expand(x.shape[0], 1)

        return self.net(torch.cat([x, t], dim=-1))

In [ ]:
def train_velocity_field(
        model,
        data,
        steps=10000
    ):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    batch_size = 512

    for step in range(steps):
        idx = torch.randint(len(data), (batch_size,), device=data.device)
        x_1 = data[idx]
        x_0 = torch.randn_like(x_1)
        t = torch.rand(batch_size, 1, device=data.device)

        x_t = (1 - t) * x_0 + t * x_1
        target_v = x_1 - x_0

        pred_v = model(x_t, t)
        loss = (pred_v - target_v).square().sum(dim=-1).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 1000 == 0:
            print(f"Step {step}: FM Loss = {loss.item():.6f}")

In [ ]:
d = data.shape[1]

model = VelocityField(d, hidden_dim=256).to(data.device)
train_velocity_field(model, data, steps=30000)

In [ ]:
candidates = torch.randn_like(data)
recon_field = EmpiricalVelocityField(candidates, learnable=True)

reconstruct(
    model,
    recon_field,
    steps=15000,
    batch_size=512,
    lr=0.01
)

In [ ]:
true_samples = data.detach()
recon_samples = recon_field.samples.detach()

dist = torch.cdist(true_samples, recon_samples)
row, col = linear_sum_assignment(dist.cpu().numpy())

errors = dist[row, col]

print("Reconstruction Mean Distance:", errors.mean().item())
print("Reconstruction Max Distance: ", errors.max().item())

In [ ]:
true = data.detach().cpu()
recon = recon_field.samples.detach().cpu()

plt.scatter(true[:, 0], true[:, 1], label="Ground Truth")
plt.scatter(recon[:, 0], recon[:, 1], marker="x", label="Reconstructed")

plt.axis("equal")
plt.legend()
plt.show()

In [ ]:
torch.manual_seed(0)

n = 100

means = torch.tensor([
    [-1.0, -1.0],
    [-1.0,  1.0],
    [ 1.0, -1.0],
    [ 1.0,  1.0]
])

data = torch.cat([
    mean + 0.3 * torch.randn(n // len(means), 2)
    for mean in means
], dim=0)

In [ ]:
d = data.shape[1]

model = VelocityField(d, hidden_dim=256).to(data.device)
train_velocity_field(model, data, steps=30000)

candidates = torch.randn_like(data)
recon_field = EmpiricalVelocityField(candidates, learnable=True)

reconstruct(
    model,
    recon_field,
    steps=15000,
    batch_size=512,
    lr=0.01
)

In [ ]:
true_samples = data.detach()
recon_samples = recon_field.samples.detach()

dist = torch.cdist(true_samples, recon_samples)
row, col = linear_sum_assignment(dist.cpu().numpy())

errors = dist[row, col]

print("Reconstruction Mean Distance:", errors.mean().item())
print("Reconstruction Max Distance: ", errors.max().item())

true = data.detach().cpu()
recon = recon_field.samples.detach().cpu()

plt.scatter(true[:, 0], true[:, 1], label="Ground Truth")
plt.scatter(recon[:, 0], recon[:, 1], marker="x", label="Reconstructed")

plt.axis("equal")
plt.legend()
plt.show()

In [ ]:
dist = torch.cdist(true, recon)
row, col = linear_sum_assignment(dist.numpy())

errors = dist[row, col]

print("Mean Distance:", errors.mean().item())
print("Max Distance: ", errors.max().item())
print("Errors > 0.5: ", (errors > 0.5).sum().item())
print("Errors > 1.0: ", (errors > 1.0).sum().item())
print("Median Distance:", errors.median().item())
print("90th Percentile:", torch.quantile(errors, 0.9).item())
print("Errors < 0.1:", (errors < 0.1).sum().item())
print("Errors < 0.2:", (errors < 0.2).sum().item())

## 8D

In [ ]:
torch.manual_seed(0)

n = 100
d = 8
std = 0.3

means = torch.zeros(4, d)
means[:, :2] = torch.tensor([
    [-1.0, -1.0],
    [-1.0,  1.0],
    [ 1.0, -1.0],
    [ 1.0,  1.0]
])

labels = torch.arange(n) % 4
data = means[labels] + std * torch.randn(n, d)

In [ ]:
d = data.shape[1]

model = VelocityField(d, hidden_dim=256).to(data.device)
train_velocity_field(model, data, steps=30000)

candidates = torch.randn_like(data)
recon_field = EmpiricalVelocityField(candidates, learnable=True)

reconstruct(
    model,
    recon_field,
    steps=15000,
    batch_size=512,
    lr=0.01
)

In [ ]:
true = data.detach().cpu()
recon = recon_field.samples.detach().cpu()

dist = torch.cdist(true, recon)
row, col = linear_sum_assignment(dist.numpy())

errors = dist[row, col]
d = true.shape[1]

print("Mean Distance:           ", errors.mean().item())
print("Normalized Mean Distance:", (errors.mean() / d**0.5).item())
print("Max Distance:            ", errors.max().item())
print("Median Distance:         ", errors.median().item())

In [ ]:
combined = torch.cat([true, recon], dim=0)

pca = PCA(n_components=2)
projected = pca.fit_transform(combined.numpy())

true_2d = projected[:len(true)]
recon_2d = projected[len(true):]

plt.figure(figsize=(6, 6))
plt.scatter(true_2d[:, 0], true_2d[:, 1], label="Ground Truth")
plt.scatter(recon_2d[:, 0], recon_2d[:, 1], marker="x", label="Reconstructed")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.axis("equal")
plt.legend()
plt.show()

## 32 D

In [ ]:
torch.manual_seed(0)

n = 100
d = 32
std = 0.3

means = torch.zeros(4, d)
means[:, :2] = torch.tensor([
    [-1.0, -1.0],
    [-1.0,  1.0],
    [ 1.0, -1.0],
    [ 1.0,  1.0]
])

labels = torch.arange(n) % 4
data = means[labels] + std * torch.randn(n, d)

In [ ]:
d = data.shape[1]

model = VelocityField(d, hidden_dim=256).to(data.device)
train_velocity_field(model, data, steps=30000)

candidates = torch.randn_like(data)
recon_field = EmpiricalVelocityField(candidates, learnable=True)

reconstruct(
    model,
    recon_field,
    steps=15000,
    batch_size=512,
    lr=0.01
)

In [ ]:
true = data.detach().cpu()
recon = recon_field.samples.detach().cpu()

dist = torch.cdist(true, recon)
row, col = linear_sum_assignment(dist.numpy())

errors = dist[row, col]
d = true.shape[1]

print("Mean Distance:           ", errors.mean().item())
print("Normalized Mean Distance:", (errors.mean() / d**0.5).item())
print("Max Distance:            ", errors.max().item())
print("Median Distance:         ", errors.median().item())

combined = torch.cat([true, recon], dim=0)

pca = PCA(n_components=2)
projected = pca.fit_transform(combined.numpy())

true_2d = projected[:len(true)]
recon_2d = projected[len(true):]

plt.figure(figsize=(6, 6))
plt.scatter(true_2d[:, 0], true_2d[:, 1], label="Ground Truth")
plt.scatter(recon_2d[:, 0], recon_2d[:, 1], marker="x", label="Reconstructed")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.axis("equal")
plt.legend()
plt.show()

In [ ]:
torch.manual_seed(0)

n = 100
d = 32
std = 0.3
scale = 2.0

means = torch.zeros(4, d)
means[:, :2] = scale * torch.tensor([
    [-1.0, -1.0],
    [-1.0,  1.0],
    [ 1.0, -1.0],
    [ 1.0,  1.0]
])

labels = torch.arange(n) % 4
data = means[labels] + std * torch.randn(n, d)

In [ ]:
d = data.shape[1]

model = VelocityField(d, hidden_dim=256).to(data.device)
train_velocity_field(model, data, steps=30000)

candidates = torch.randn_like(data)
recon_field = EmpiricalVelocityField(candidates, learnable=True)

reconstruct(
    model,
    recon_field,
    steps=15000,
    batch_size=512,
    lr=0.01
)

In [ ]:
true = data.detach().cpu()
recon = recon_field.samples.detach().cpu()

dist = torch.cdist(true, recon)
row, col = linear_sum_assignment(dist.numpy())

errors = dist[row, col]
d = true.shape[1]

print("Mean Distance:           ", errors.mean().item())
print("Normalized Mean Distance:", (errors.mean() / d**0.5).item())
print("Max Distance:            ", errors.max().item())
print("Median Distance:         ", errors.median().item())

combined = torch.cat([true, recon], dim=0)

pca = PCA(n_components=2)
projected = pca.fit_transform(combined.numpy())

true_2d = projected[:len(true)]
recon_2d = projected[len(true):]

plt.figure(figsize=(6, 6))
plt.scatter(true_2d[:, 0], true_2d[:, 1], label="Ground Truth")
plt.scatter(recon_2d[:, 0], recon_2d[:, 1], marker="x", label="Reconstructed")

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.axis("equal")
plt.legend()
plt.show()